# A/B Testing for Analysts: Design, Sample Size, Analysis and Pitfalls

Notebook for **Chapter 13** of the *analyst* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/analyst/ch13.html

Tip: change a number, predict the output, then run the cell again.

## 13.1 Designing a Test

## 13.2 Sample Size and Power

In [ ]:
import numpy as np
from scipy.stats import norm
def n_per_group(p1,p2,alpha=0.05,power=0.8):
    za=norm.ppf(1-alpha/2); zb=norm.ppf(power)
    pbar=(p1+p2)/2
    num=(za*np.sqrt(2*pbar*(1-pbar))+zb*np.sqrt(p1*(1-p1)+p2*(1-p2)))**2
    return num/(p2-p1)**2
for p1,p2 in [(0.05,0.06),(0.05,0.055),(0.10,0.11),(0.05,0.0625)]:
    print(p1,p2,round(n_per_group(p1,p2),1))

0.05 0.06 8157.7
0.05 0.055 31233.4
0.1 0.11 14750.8
0.05 0.0625 5332.1


## 13.3 Analysing the Result

In [ ]:
import numpy as np
from scipy import stats
nC,xC=8200,410; nT,xT=8200,492
pC,pT=xC/nC,xT/nT
pp=(xC+xT)/(nC+nT)
se_pool=np.sqrt(pp*(1-pp)*(1/nC+1/nT))
z=(pT-pC)/se_pool
p=2*(1-stats.norm.cdf(abs(z)))
se=np.sqrt(pC*(1-pC)/nC+pT*(1-pT)/nT)
lo,hi=(pT-pC)-1.96*se,(pT-pC)+1.96*se
print(round(se_pool,5),round(z,3),round(p,4),round(lo,4),round(hi,4))

0.00356 2.809 0.005 0.003 0.017


In [ ]:
rng=np.random.default_rng(7)
c=rng.exponential(300,5000)*(rng.random(5000)<0.2)
t=rng.exponential(300,5000)*(rng.random(5000)<0.2)*1.0
t=t*1.06
print(round(c.mean(),2),round(t.mean(),2),stats.ttest_ind(t,c,equal_var=False))
print(stats.mannwhitneyu(t,c))

60.14 63.31 TtestResult(statistic=np.float64(0.8552077181849908), pvalue=np.float64(0.39245672014246635), df=np.float64(9967.322625953564))
MannwhitneyuResult(statistic=np.float64(12522922.0), pvalue=np.float64(0.8198981241835632))


## 13.4 Pitfalls and Beyond the Basic Test

In [ ]:
print(stats.chisquare([50400,49600],[50000,50000]))
print(stats.chisquare([50800,49200],[50000,50000]))

Power_divergenceResult(statistic=np.float64(6.4), pvalue=np.float64(0.01141203638600166))
Power_divergenceResult(statistic=np.float64(25.6), pvalue=np.float64(4.200393976022009e-07))


In [ ]:
import pandas as pd
d=pd.DataFrame({"device":["mobile","mobile","desktop","desktop"],"arm":["A","B","A","B"],"users":[1000,4000,4000,1000],"conv":[50,220,400,110]})
d["rate_%"]=(d.conv/d.users*100).round(2)
print(d)
t=d.groupby("arm")[["users","conv"]].sum(); t["rate_%"]=(t.conv/t.users*100).round(2); print(t)

    device arm  users  conv  rate_%
0   mobile   A  1000    50     5.0
1   mobile   B  4000   220     5.5
2  desktop   A  4000   400    10.0
3  desktop   B  1000   110    11.0
     users  conv  rate_%
arm                     
A     5000   450     9.0
B     5000   330     6.6


## 13.5 Resampling, Corrections, CUPED and Ratio Metrics

In [ ]:
import numpy as np
rng = np.random.default_rng(7)
a = rng.lognormal(5.0, 1.0, 300).round(0)
b = rng.lognormal(5.15, 1.0, 300).round(0)

def boot(x, stat, n=5000, seed=1):
    r = np.random.default_rng(seed)
    idx = r.integers(0, len(x), size=(n, len(x)))      # n resamples, each len(x) draws with replacement
    return stat(x[idx], axis=1)

print("means  A, B:", a.mean().round(1), b.mean().round(1))
print("medians A, B:", np.median(a), np.median(b))
d_mean = boot(b, np.mean) - boot(a, np.mean)
d_med = boot(b, np.median) - boot(a, np.median)
print("mean difference", round(b.mean() - a.mean(), 1), "95% CI", np.percentile(d_mean, [2.5, 97.5]).round(1))
print("median difference", np.median(b) - np.median(a), "95% CI", np.percentile(d_med, [2.5, 97.5]).round(1))

means  A, B: 194.3 232.5
medians A, B: 138.0 150.0
mean difference 38.2 95% CI [ 1.1 76.1]
median difference 12.0 95% CI [-6.5 35. ]


In [ ]:
import numpy as np
from scipy import stats
rng = np.random.default_rng(7)
a = rng.lognormal(5.0, 1.0, 300).round(0)
b = rng.lognormal(5.15, 1.0, 300).round(0)
observed = b.mean() - a.mean()
pool = np.concatenate([a, b])
r = np.random.default_rng(2)
N = 10000
count = 0
for _ in range(N):
    p = r.permutation(pool)
    if abs(p[300:].mean() - p[:300].mean()) >= abs(observed):
        count += 1
print("permutation p:", round((count + 1) / (N + 1), 4))
print("Welch t-test p:", round(stats.ttest_ind(b, a, equal_var=False).pvalue, 4))
print("Mann-Whitney p:", round(stats.mannwhitneyu(b, a).pvalue, 4))

permutation p: 0.0415
Welch t-test p: 0.044
Mann-Whitney p: 0.1013


In [ ]:
import numpy as np
p = np.array([0.001, 0.008, 0.012, 0.030, 0.041, 0.049, 0.20, 0.45, 0.62, 0.90])   # already sorted
m = len(p)
limit = 0.05 * np.arange(1, m + 1) / m
print("BH limits:", limit.round(3))
passes = p <= limit
k = np.max(np.where(passes)[0]) + 1
print("BH keeps the first", k, "| Bonferroni keeps", int((p <= 0.05 / m).sum()), "| no correction keeps", int((p < 0.05).sum()))

BH limits: [0.005 0.01  0.015 0.02  0.025 0.03  0.035 0.04  0.045 0.05 ]
BH keeps the first 3 | Bonferroni keeps 1 | no correction keeps 6


In [ ]:
import numpy as np
rng = np.random.default_rng(5)
n = 2000
pre = rng.gamma(4, 50, size=2 * n)                     # spend before the test
y = 0.8 * pre + rng.normal(0, 60, size=2 * n)          # spend during the test, strongly related to the past
arm = np.r_[np.zeros(n), np.ones(n)]
y = y + 8 * arm                                        # the true lift is 8

def lift_and_se(v):
    a, b = v[arm == 0], v[arm == 1]
    return float(round(b.mean() - a.mean(), 2)), float(round(np.sqrt(a.var(ddof=1) / n + b.var(ddof=1) / n), 2))

r = np.corrcoef(y, pre)[0, 1]
theta = np.cov(y, pre)[0, 1] / pre.var(ddof=1)
y_cuped = y - theta * (pre - pre.mean())
print("correlation:", round(r, 3), "theta:", round(theta, 3))
print("raw    lift, se:", lift_and_se(y))
print("CUPED  lift, se:", lift_and_se(y_cuped))
print("variance ratio:", round(y_cuped.var() / y.var(), 3), "vs 1 - r^2:", round(1 - r ** 2, 3))

correlation: 0.8 theta: 0.815
raw    lift, se: (1.84, 3.21)
CUPED  lift, se: (6.44, 1.92)
variance ratio: 0.36 vs 1 - r^2: 0.36


In [ ]:
import numpy as np
rng = np.random.default_rng(9)
users = 500
sess = rng.poisson(4, users) + 1                   # sessions per user
lam = rng.gamma(2, 0.75, users)                    # each user's own clicks-per-session rate
user_id = np.repeat(np.arange(users), sess)        # one row per session
clicks_row = rng.poisson(lam[user_id])
clicks_user = np.bincount(user_id, weights=clicks_row)

ratio = clicks_row.mean()                          # total clicks / total sessions
naive_se = clicks_row.std(ddof=1) / np.sqrt(len(clicks_row))
mx = sess.mean()
vx, vy = sess.var(ddof=1), clicks_user.var(ddof=1)
cxy = np.cov(sess, clicks_user)[0, 1]
delta_se = np.sqrt((vy - 2 * ratio * cxy + ratio ** 2 * vx) / (users * mx ** 2))
r2 = np.random.default_rng(1)
boot = []
for _ in range(4000):
    i = r2.integers(0, users, users)               # resample USERS
    boot.append(clicks_user[i].sum() / sess[i].sum())
print("sessions:", len(clicks_row), " ratio:", round(ratio, 3))
print("naive se (sessions independent):", round(naive_se, 4))
print("delta-method se:", round(delta_se, 4))
print("user-level bootstrap se:", round(np.std(boot), 4))

sessions: 2557  ratio: 1.455
naive se (sessions independent): 0.0321
delta-method se: 0.0586
user-level bootstrap se: 0.058


In [ ]:
import numpy as np
from scipy import stats
rs = np.random.default_rng(3)
xa, na, xb, nb = 120, 2400, 150, 2400                 # A: 5.0 percent, B: 6.25 percent
sa = rs.beta(1 + xa, 1 + na - xa, 200000)
sb = rs.beta(1 + xb, 1 + nb - xb, 200000)
print("P(B > A):", round((sb > sa).mean(), 3))
print("95% credible interval for B - A:", np.percentile(sb - sa, [2.5, 97.5]).round(4))
print("frequentist chi-square p:", round(stats.chi2_contingency([[xa, na - xa], [xb, nb - xb]], correction=False)[1], 4))

P(B > A): 0.969
95% credible interval for B - A: [-0.0006  0.0256]
frequentist chi-square p: 0.0602


## 13.6 Causal Analysis Without an Experiment

In [ ]:
import numpy as np, pandas as pd
rng = np.random.default_rng(25)
rows = []
for s in range(40):
    treated = int(s < 20)
    base = 100 + 10 * treated + rng.normal(0, 6)            # treated stores are bigger to begin with
    for m in range(12):
        post = int(m >= 6)
        sales = base + 1.5 * m + 12 * treated * post + rng.normal(0, 4)   # common trend 1.5 a month, true effect 12
        rows.append((s, m, treated, post, sales))
df = pd.DataFrame(rows, columns=["store", "month", "treated", "post", "sales"])

g = df.groupby(["treated", "post"])["sales"].mean().unstack()
print(g.round(2))
did = (g.loc[1, 1] - g.loc[1, 0]) - (g.loc[0, 1] - g.loc[0, 0])
print("DiD from group means:", round(did, 2))
print("naive after comparison:", round(g.loc[1, 1] - g.loc[0, 1], 2))

X = np.column_stack([np.ones(len(df)), df.treated, df.post, df.treated * df.post])
beta = np.linalg.lstsq(X, df.sales.values, rcond=None)[0]
print("regression coefficients:", beta.round(2))

r = np.random.default_rng(1)
groups = {s: d for s, d in df.groupby("store")}
est = []
for _ in range(1000):                                        # resample whole stores
    pick = r.choice(40, size=40)
    d = pd.concat([groups[s].assign(store=i) for i, s in enumerate(pick)])
    gg = d.groupby(["treated", "post"])["sales"].mean()
    est.append((gg[1, 1] - gg[1, 0]) - (gg[0, 1] - gg[0, 0]))
print("95% interval (store bootstrap):", np.percentile(est, [2.5, 97.5]).round(2))

post          0       1
treated                
0        103.04  111.70
1        114.36  135.16
DiD from group means: 12.15
naive after comparison: 23.47
regression coefficients: [103.04  11.32   8.65  12.15]
95% interval (store bootstrap): [10.58 13.72]


In [ ]:
import numpy as np, pandas as pd
rng = np.random.default_rng(25)
rows = []
for s in range(40):
    treated = int(s < 20)
    base = 100 + 10 * treated + rng.normal(0, 6)            # treated stores are bigger to begin with
    for m in range(12):
        post = int(m >= 6)
        sales = base + 1.5 * m + 12 * treated * post + rng.normal(0, 4)   # common trend 1.5 a month, true effect 12
        rows.append((s, m, treated, post, sales))
df = pd.DataFrame(rows, columns=["store", "month", "treated", "post", "sales"])

pre = df[df.post == 0]
for t in (0, 1):
    d = pre[pre.treated == t].groupby("month")["sales"].mean()
    print("pre-period slope, treated =", t, ":", round(np.polyfit(d.index, d.values, 1)[0], 2), "per month")
pre = pre.assign(fake=(pre.month >= 3).astype(int))
gp = pre.groupby(["treated", "fake"])["sales"].mean().unstack()
print("placebo DiD:", round((gp.loc[1, 1] - gp.loc[1, 0]) - (gp.loc[0, 1] - gp.loc[0, 0]), 2))

pre-period slope, treated = 0 : 1.58 per month
pre-period slope, treated = 1 : 1.7 per month
placebo DiD: 0.22


In [ ]:
import pandas as pd
d = pd.DataFrame({
    "past":  ["low", "low", "high", "high"],
    "app":   [1, 0, 1, 0],
    "n":     [200, 800, 800, 200],
    "mean_spend": [120, 100, 320, 300]})
tot = d.assign(total=d.n * d.mean_spend).groupby("app")[["n", "total"]].sum()
naive = tot.loc[1, "total"] / tot.loc[1, "n"] - tot.loc[0, "total"] / tot.loc[0, "n"]
print("naive difference:", naive)
w = d.groupby("past")["n"].sum() / d["n"].sum()                       # weights: share of customers in each stratum
diff = d.pivot(index="past", columns="app", values="mean_spend")
diff["effect"] = diff[1] - diff[0]
print(diff)
print("stratified effect:", (diff["effect"] * w).sum())

naive difference: 140.0
app     0    1  effect
past                  
high  300  320      20
low   100  120      20
stratified effect: 20.0


In [ ]:
import numpy as np
rng = np.random.default_rng(4)
score = rng.uniform(500, 700, 4000)
y = 50 + 0.10 * (score - 600) + 8 * (score >= 600) + rng.normal(0, 5, 4000)   # true jump = 8
for bw in (50, 20):
    m = np.abs(score - 600) <= bw
    below, above = m & (score < 600), m & (score >= 600)
    fb = np.polyfit(score[below] - 600, y[below], 1)
    fa = np.polyfit(score[above] - 600, y[above], 1)
    print("bandwidth", bw, "jump at 600:", round(fa[1] - fb[1], 2), "n =", int(m.sum()))

bandwidth 50 jump at 600: 7.77 n = 1981
bandwidth 20 jump at 600: 7.12 n = 799


## Chapter recap

## Practice set

Problem 1 (easy). Control 200 of 4,000 convert, treatment 240 of 4,000. What are the two rates and the difference? **Answer:** 5.0% and 6.0%, a difference of 1.0 point (a 20% relative lift).

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/analyst/ch13.html